# Water Quality (Potability) Classification using Logistic Regression

**Goal:** Predict whether water is safe to drink (`Potability` = 1) or not (`Potability` = 0) from 9 chemical/physical properties.

**Type of problem:** Binary classification  
**Model:** Logistic Regression inside a scikit-learn Pipeline

**Pipeline:** Load data → Understand → Check quality → Visualize → Split → Scale → Train → Predict → Evaluate → Explain

## Step 1: Import libraries
We need pandas/numpy for data, matplotlib for plots, and scikit-learn for the model and metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)

print("libraries imported successfully")

## Step 2: Load the dataset
The dataset has 20,000 water samples. Each row is one sample; the last column `Potability` is the answer we want to predict.

In [ ]:
df = pd.read_csv("water_quality_potability_synthetic_20000.csv")
print(df.head())

## Step 3: Understand the dataset
| Column | Meaning |
|---|---|
| ph | Acidity / alkalinity of water |
| Hardness | Calcium and magnesium salts (mg/L) |
| Solids | Total dissolved solids (ppm) |
| Chloramines | Disinfectant level (ppm) |
| Sulfate | Dissolved sulfate (mg/L) |
| Conductivity | Electrical conductivity (µS/cm) |
| Organic_carbon | Organic carbon (ppm) |
| Trihalomethanes | By-product of chlorination (µg/L) |
| Turbidity | Cloudiness of water (NTU) |
| **Potability** | **Target: 1 = safe, 0 = not safe** |

In [ ]:
print("shape (rows, columns):", df.shape)
print("column names:", list(df.columns))
print("\ndata types and non-null counts:")
df.info()
print("\nstatistical summary:")
print(df.describe())

## Step 4: Data quality checks
Before training we must check three things: missing values, duplicate rows, and class balance.

In [ ]:
print("missing values:")
print(df.isnull().sum())

print("\nduplicate rows:", df.duplicated().sum())

print("\nclass distribution:")
print(df["Potability"].value_counts())
print("\nclass percentage:")
print((df["Potability"].value_counts(normalize=True) * 100).round(2))

**Observation:** There are no missing values, but the data is **highly imbalanced**: only about 6.7% of samples are potable.
A model that always says "not safe" would already get ~93% accuracy, so **accuracy alone is misleading** here.
We will handle this with `class_weight="balanced"` and judge the model with precision, recall, F1 and ROC-AUC.

## Step 5: Visualize the data

In [ ]:
# 5a. Class distribution
df["Potability"].value_counts().sort_index().plot(
    kind="bar", rot=0, title="Class Distribution (0 = Not Safe, 1 = Safe)")
plt.xlabel("Potability")
plt.ylabel("Number of Samples")
plt.show()

In [ ]:
# 5b. Feature distributions for each class
features = df.columns.drop("Potability")
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, col in zip(axes.ravel(), features):
    df[df["Potability"] == 0][col].plot(kind="hist", bins=30, alpha=0.5, density=True, ax=ax, label="Not safe")
    df[df["Potability"] == 1][col].plot(kind="hist", bins=30, alpha=0.5, density=True, ax=ax, label="Safe")
    ax.set_title(col)
    ax.set_ylabel("")
axes[0, 0].legend()
plt.tight_layout()
plt.show()

**Observation:** Safe water tends to have lower Solids, Chloramines, Turbidity and Trihalomethanes. These differences are what the model will learn.

In [ ]:
# 5c. Correlation heatmap
corr = df.corr()
plt.figure(figsize=(8, 6))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(corr)), corr.columns, rotation=90)
plt.yticks(range(len(corr)), corr.columns)
plt.title("Correlation Heatmap")
plt.show()

## Step 6: Separate features (X) and target (y)
`X` = inputs the model sees, `y` = the answer it must predict.

In [ ]:
X = df.drop("Potability", axis=1)
y = df["Potability"]

print("Features (X):")
print(X.head())
print("\nTarget (y):")
print(y.head())

## Step 7: Train-test split
We keep 80% of data for training and hide 20% for testing, so we can check how the model performs on unseen data.  
`stratify=y` keeps the same safe/not-safe ratio in both parts (very important for imbalanced data).  
`random_state=42` makes the result reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("training records:", len(X_train))
print("testing records:", len(X_test))
print("\nsafe water % in train:", round(y_train.mean() * 100, 2))
print("safe water % in test :", round(y_test.mean() * 100, 2))

## Step 8: Feature scaling and model creation (Pipeline)
- **StandardScaler:** Features have very different ranges (Solids ~ 25000, Turbidity ~ 7). Scaling puts all of them on the same scale (mean 0, std 1) so the coefficients can be compared fairly.
- **LogisticRegression:** Calculates the probability that a sample is potable.
- **class_weight="balanced":** Gives more importance to the rare class (safe water) so the model does not ignore it.
- **Pipeline:** Chains scaling + model so the scaler is learned only from training data (no data leakage).

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=1000))
])
print(model)

## Step 9: Train the model (fit)

In [ ]:
model.fit(X_train, y_train)
print("model trained successfully")

## Step 10: Predict on test data

In [ ]:
y_pred = model.predict(X_test)

comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})
print(comparison.head(15))

## Step 11: Confusion matrix
- Rows = actual class, columns = predicted class
- `[0,0]` True Negative: correctly said not safe
- `[0,1]` False Positive: said safe, but actually not safe (**dangerous mistake**)
- `[1,0]` False Negative: said not safe, but actually safe
- `[1,1]` True Positive: correctly said safe

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print("rows = actual, columns = predicted")
print(cm)

plt.imshow(cm, cmap="Blues")
plt.colorbar()
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=14)
plt.xticks([0, 1], ["Not Safe", "Safe"])
plt.yticks([0, 1], ["Not Safe", "Safe"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

## Step 12: Classification report

In [ ]:
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["Not Safe", "Safe"],
    zero_division=0
))

## Step 13: Evaluation metrics
- **Accuracy:** overall correct predictions
- **Precision:** of all samples predicted *safe*, how many were really safe
- **Recall:** of all really *safe* samples, how many did we find
- **F1 score:** balance between precision and recall

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("Accuracy :", round(accuracy, 3))
print("Precision:", round(precision, 3))
print("Recall   :", round(recall, 3))
print("F1 Score :", round(f1, 3))

**Note:** With imbalanced data, accuracy looks lower than a naive model's 93%, but recall for the safe class is much higher. That is a better trade-off for a water-safety problem.

## Step 14: Understand prediction probabilities
Logistic Regression does not just give 0 or 1; it gives a probability.  
By default, probability ≥ 0.5 → predicted class 1 (Safe).

In [ ]:
probabilities = model.predict_proba(X_test)
print("columns are ordered as classes:", model.classes_)
print(probabilities[:10].round(3))

## Step 15: ROC curve and ROC-AUC
ROC-AUC measures how well the model separates the two classes at *all* thresholds.  
1.0 = perfect, 0.5 = random guessing.

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)
fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve")
plt.legend()
plt.grid(True)
plt.show()

print("ROC-AUC:", round(auc, 3))

## Step 16: Threshold tuning
The default threshold of 0.5 is not always best. A higher threshold makes the model stricter about calling water "safe" (higher precision, lower recall).  
For drinking water, wrongly calling unsafe water "safe" is risky, so a stricter threshold can be preferred.

In [ ]:
rows = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    pred_t = (y_prob >= t).astype(int)
    rows.append({
        "threshold": t,
        "precision": precision_score(y_test, pred_t, zero_division=0),
        "recall": recall_score(y_test, pred_t, zero_division=0),
        "f1": f1_score(y_test, pred_t, zero_division=0),
    })
print(pd.DataFrame(rows).round(3).to_string(index=False))

## Step 17: Explainability: which features matter?
In Logistic Regression each feature has a **coefficient**:
- Positive coefficient → higher value increases the chance of *safe*
- Negative coefficient → higher value decreases the chance of *safe*
- Bigger absolute value → stronger effect (valid to compare because we scaled the features)

**Odds ratio** = exp(coefficient). Example: 0.5 means "one standard deviation increase halves the odds of being safe".

In [ ]:
coefs = model.named_steps["classifier"].coef_[0]
importance = pd.DataFrame({
    "feature": X.columns,
    "coefficient": coefs,
    "odds_ratio": np.exp(coefs)
}).sort_values("coefficient")

print(importance.round(3).to_string(index=False))

plt.figure(figsize=(8, 5))
colors = ["tomato" if c < 0 else "seagreen" for c in importance["coefficient"]]
plt.barh(importance["feature"], importance["coefficient"], color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Coefficient (effect on log-odds of Safe)")
plt.title("Feature Importance (Logistic Regression)")
plt.show()

**Interpretation:** Solids, Chloramines and Turbidity have the strongest negative effect, so water with high values of these is less likely to be safe.

## Step 18: Predict for a new water sample

In [ ]:
new_sample = pd.DataFrame({
    "ph": [7.2],
    "Hardness": [180],
    "Solids": [7000],
    "Chloramines": [4.0],
    "Sulfate": [250],
    "Conductivity": [380],
    "Organic_carbon": [10],
    "Trihalomethanes": [48],
    "Turbidity": [2.3],
})

predicted_class = model.predict(new_sample)[0]
predicted_probability = model.predict_proba(new_sample)[0]

if predicted_class == 1:
    print("The water is predicted to be SAFE to drink.")
else:
    print("The water is predicted to be NOT safe to drink.")

print("class order:", model.classes_)
print("Predicted probabilities [Not Safe, Safe]:", predicted_probability.round(3))

## Step 19: Conclusion
- The dataset is highly imbalanced (~6.7% safe), so accuracy alone is misleading.
- Using `class_weight="balanced"` greatly improves recall for safe water.
- ROC-AUC shows that the model separates safe and unsafe water reasonably well.
- Explainability: low Solids, Chloramines, Turbidity, Sulfate and Organic carbon are linked with safe water.
- **Limitation:** Dataset is synthetic and Logistic Regression is linear. Models like Random Forest may capture more complex patterns.